# P5-5 · GĐ 5 · Fine-tune LoRA

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q peft

## Lý thuyết

In [ ]:
import torch, torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base.requires_grad_(False)                       # đóng băng W
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))     # B = 0: ban đầu không đổi gì
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)

layer = LoRALinear(nn.Linear(320, 320), r=4, alpha=8)
x = torch.randn(3, 320)
print("tham số được train:", sum(p.numel() for p in layer.parameters() if p.requires_grad))   # 4·(320+320) = 2560
print("lúc đầu giống hệt lớp gốc:", torch.allclose(layer(x), layer.base(x)))               # vì B = 0

tham số được train: 2560
lúc đầu giống hệt lớp gốc: True


## Bài tập 4: Tự viết lớp LoRA {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base, self.scale = base, alpha / r
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.merged = False

    def forward(self, x):
        out = self.base(x)
        return out if self.merged else out + self.scale * (x @ self.A.T @ self.B.T)

    @torch.no_grad()
    def merge(self):
        self.base.weight += self.scale * self.B @ self.A
        self.merged = True

base = nn.Linear(768, 768)
x = torch.randn(4, 768)
lora = LoRALinear(base)
print("lúc đầu bằng gốc:", torch.allclose(lora(x), base(x)))
with torch.no_grad():
    lora.B.normal_(0, 0.01)                    # giả lập đã train
before = lora(x)
lora.merge()
print("sau merge không đổi:", torch.allclose(before, lora(x), atol=1e-5))
print("tham số train:", sum(p.numel() for p in lora.parameters() if p.requires_grad))

lúc đầu bằng gốc: True
sau merge không đổi: True
tham số train: 12288


## Bài tập 5: LoRA học được thay đổi hạng thấp {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
d = 64
W0 = torch.randn(d, d) / d ** 0.5
dW = (torch.randn(d, 2) @ torch.randn(2, d)) * 0.3          # thay đổi cần học: hạng 2
X = torch.randn(2000, d); Y = X @ (W0 + dW).T

class LoRALinear(nn.Module):
    def __init__(self, W0, r):
        super().__init__()
        self.W0 = nn.Parameter(W0.clone(), requires_grad=False)
        # TODO: A (r × d, khởi tạo nhỏ ngẫu nhiên), B (d × r, khởi tạo 0)
    def forward(self, x):
        return x @ self.W0.T   # TODO: cộng nhánh LoRA

def train(r, steps=800):
    torch.manual_seed(1)
    m = LoRALinear(W0, r)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-2)
    for _ in range(steps):
        i = torch.randint(0, 2000, (128,))
        loss = ((m(X[i]) - Y[i]) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item(), sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, W0, r):
        super().__init__()
        self.W0 = nn.Parameter(W0.clone(), requires_grad=False)
        self.A = nn.Parameter(torch.randn(r, d) * 0.01)
        self.B = nn.Parameter(torch.zeros(d, r))
    def forward(self, x):
        return x @ self.W0.T + x @ self.A.T @ self.B.T

kq = {r: train(r) for r in (1, 2, 8)}
for r, (l, n) in kq.items():
    print(f"rank {r}: loss {l:.5f}, train {n} tham số ({n / (d * d):.1%} so với toàn bộ)")

rank 1: loss 3.77211, train 128 tham số (3.1% so với toàn bộ)
rank 2: loss 0.00000, train 256 tham số (6.2% so với toàn bộ)
rank 8: loss 0.00001, train 1024 tham số (25.0% so với toàn bộ)


## Bài tập 6: Gắn LoRA vào attention bằng PEFT và gộp lại {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch
from transformers import BertConfig, BertModel
from peft import LoraConfig, get_peft_model
torch.manual_seed(0)
base = BertModel(BertConfig(vocab_size=200, hidden_size=64, num_hidden_layers=2, num_attention_heads=4, intermediate_size=128)).eval()
x = torch.randint(0, 200, (2, 10))
# TODO: model = get_peft_model(base, LoraConfig(r=4, lora_alpha=8, target_modules=["query", "value"]))
#       đếm tham số; gán lora_B ngẫu nhiên; out_lora; merged = model.merge_and_unload(); out_merged

In [ ]:
model = get_peft_model(base, LoraConfig(r=4, lora_alpha=8, target_modules=["query", "value"]))
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(f"train {n_train:,} / {n_all:,} tham số ({n_train / n_all:.2%})")
with torch.no_grad():
    for name, p in model.named_parameters():
        if "lora_B" in name:
            p.normal_(0, 0.1)
    out_lora = model(x).last_hidden_state
    merged = model.merge_and_unload()
    out_merged = merged(x).last_hidden_state
chenh = (out_lora - out_merged).abs().max().item()
print("chênh lệch sau khi gộp:", chenh, "| còn lớp LoRA?", any("lora" in n for n, _ in merged.named_parameters()))

train 2,048 / 118,976 tham số (1.72%)
chênh lệch sau khi gộp: 2.384185791015625e-07 | còn lớp LoRA? False


## Bài tập 7: Kiểm tra bộ dữ liệu trước khi train DreamBooth {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
ds = [("b01.jpg", (1024, 1024), "a1", "ảnh sks_ban màu trắng trong phòng ngủ"), ("b02.jpg", (800, 1200), "a2", "ảnh sks_ban nhìn nghiêng"),
      ("b03.jpg", (480, 640), "a3", "ảnh sks_ban màu hồng"), ("b04.jpg", (1024, 768), "a1", "ảnh sks_ban màu trắng trong phòng ngủ"),
      ("b05.jpg", (1024, 1024), "a5", "ảnh bàn trang điểm có gương"), ("b06.jpg", (2048, 1536), "a6", "ảnh sks_ban"),
      ("b07.jpg", (1024, 1024), "a7", "ảnh sks_ban"), ("b08.jpg", (900, 900), "a8", "ảnh sks_ban"),
      ("b09.jpg", (1024, 1024), "a9", "ảnh sks_ban cạnh cửa sổ, ánh sáng tự nhiên"), ("b10.jpg", (512, 384), "b0", "ảnh sks_ban màu gỗ"),
      ("b11.jpg", (1200, 1600), "b1", "ảnh sks_ban có ghế đôn"), ("b12.jpg", (1024, 1024), "b2", "ảnh sks_ban")]
TU = "sks_ban"

def kiem_tra(ds):
    # TODO: trả về dict {"nho": [...], "trung": [...], "thieu_tu": [...], "ti_le_chu_thich_lap": số}
    return {}

In [ ]:
from collections import Counter

def kiem_tra(ds):
    da_gap, trung = set(), []
    for ten, _, h, _ in ds:
        if h in da_gap:
            trung.append(ten)
        else:
            da_gap.add(h)
    dem = Counter(c for *_, c in ds)
    return {"nho": [t for t, (w, h), *_ in ds if min(w, h) < 512],
            "trung": trung,
            "thieu_tu": [t for t, _, _, c in ds if TU not in c],
            "ti_le_chu_thich_lap": sum(n for n in dem.values() if n > 1) / len(ds)}

bc = kiem_tra(ds)
for k, v in bc.items():
    print(f"{k:22s}: {v}")

nho                   : ['b03.jpg', 'b10.jpg']
trung                 : ['b04.jpg']
thieu_tu              : ['b05.jpg']
ti_le_chu_thich_lap   : 0.5
